In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import scipy.io as scp
import numpy as np
import matplotlib.pyplot as plt

from torch_geometric.nn import GCNConv
from torch_geometric.utils import from_scipy_sparse_matrix
from sklearn.metrics import roc_auc_score

Load the data

In [6]:
data = scp.loadmat('ACM.mat')
print(list(data.keys()))

['__header__', '__version__', '__globals__', 'Network', 'Label', 'Attributes', 'Class']


Create the Graph Autoencoder class and the loss function

In [7]:
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(Encoder, self).__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, latent_dim)

    def forward(self, x, edge):
        x = self.conv1(x, edge)
        x = torch.relu(x)

        x = self.conv2(x, edge)
        x = torch.relu(x)
        return x

class AtrDecoder(nn.Module):
    """
    Attribute decoder for graph
    """
    def __init__(self, latent_dim, hidden_dim, output_dim):
        super(AtrDecoder, self).__init__()
        self.conv1 = GCNConv(latent_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, output_dim)
    
    def forward(self, z, edge):
        x = self.conv1(z, edge)
        x = torch.relu(x)
        
        x = self.conv2(x, edge)
        return x

class Decoder(nn.Module):
    """
    Z @ Z^T reconstruction of the adjacency matrix.
    """
    def __init__(self, latent_dim):
        super(Decoder, self).__init__()
        self.conv = GCNConv(latent_dim, latent_dim)
    
    def forward(self, z, edge):

        z = self.conv(z, edge)
        z = torch.relu(z)
        
        A_hat = torch.matmul(z, z.t())
        
        return A_hat


class GraphAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(GraphAutoencoder, self).__init__()
        self.encoder = Encoder(input_dim, hidden_dim, latent_dim)
        self.atr_dec = AtrDecoder(latent_dim, hidden_dim, input_dim)
        self.dec = Decoder(latent_dim)
    
    def forward(self, x, edge_index):
        # Encode
        z = self.encoder(x, edge_index)
        
        # Decode attributes
        x_hat = self.atr_dec(z, edge_index)
        
        # Decode
        A_hat = self.dec(z, edge_index)
        
        return x_hat, A_hat
    
def loss(X, x_hat, A, A_hat, alpha):
    # Frobenius norm squared 
    attr_loss = torch.norm(X - x_hat, p="fro") ** 2
    struct_loss = torch.norm(A - A_hat, p="fro") ** 2

    # Combined loss
    total_loss = alpha * attr_loss + (1 - alpha) * struct_loss

    return total_loss, attr_loss, struct_loss

In [ ]:
X = data['Attributes'].toarray()
A_sparse = data['Network']
labels = data['Label'].flatten()


edge, edge_weight = from_scipy_sparse_matrix(A_sparse)

X_tensor = torch.FloatTensor(X)
labels_tensor = torch.LongTensor(labels)

### Test data loading and find knowledge about the dataset
print(f"Nr. nodes: {X.shape[0]}")
print(f"Nr. features: {X.shape[1]}")
print(f"Nr. edges: {edge.shape[1]}")
print(f"Nr. of anomalies: {labels.sum()}")
print(f"Anomaly percentage: {labels.sum() / len(labels):.2%}")

Nr. nodes: 16484
Nr. features: 8337
Nr. edges: 164350
Nr. of anomalies: 597
Anomaly percentage: 3.62%


In [11]:
import tqdm

input_dim = X.shape[1]
model = GraphAutoencoder(input_dim=input_dim, hidden_dim=128, latent_dim=64)
optimizer = optim.Adam(model.parameters(), lr=0.004)
A_dense = torch.FloatTensor(A_sparse.toarray())
alpha = 0.8

num_epochs = 50

loss_history = []
attr_loss_history = []
struct_loss_history = []
roc_auc_history = []
epoch_roc_auc = []

for epoch in tqdm.tqdm(range(num_epochs)):
    model.train()
    optimizer.zero_grad()

    X_hat, A_hat = model(X_tensor, edge)

    total_loss, attr_loss, struct_loss = loss(
        X_tensor, X_hat, A_dense, A_hat, alpha
    )

    total_loss.backward()

    # update weights
    optimizer.step()

    # Store loss values
    loss_history.append(total_loss.item())
    attr_loss_history.append(attr_loss.item())
    struct_loss_history.append(struct_loss.item())

    # compute ROC AUC every 5 epochs
    if (epoch + 1) % 5 == 0:
        model.eval()
        with torch.no_grad():
            X_hat_eval, A_hat_eval = model(X_tensor, edge)

            attr_errors = torch.norm(X_tensor - X_hat_eval, p=2, dim=1).numpy()

            # Structure reconstruction error (per node)
            A_errors = torch.norm(A_dense - A_hat_eval, p=2, dim=1).numpy()

            # Combined reconstruction error
            reconstruction_errors = alpha * attr_errors + (1 - alpha) * A_errors

            roc_auc = roc_auc_score(labels, reconstruction_errors)
            roc_auc_history.append(roc_auc)
            epoch_roc_auc.append(epoch + 1)

            print(
                f"Epoch {epoch+1:3d}/{num_epochs} | "
                f"Loss: {total_loss.item():.4f} | "
                f"Attr Loss: {attr_loss.item():.4f} | "
                f"Struct Loss: {struct_loss.item():.4f} | "
                f"ROC AUC: {roc_auc:.4f}"
            )

 10%|█         | 5/50 [00:26<04:11,  5.60s/it]

Epoch   5/50 | Loss: 40074.7891 | Attr Loss: 9040.5605 | Struct Loss: 164211.7031 | ROC AUC: 0.8951


 20%|██        | 10/50 [00:53<03:45,  5.64s/it]

Epoch  10/50 | Loss: 39588.6914 | Attr Loss: 8501.1943 | Struct Loss: 163938.6719 | ROC AUC: 0.8949


 30%|███       | 15/50 [01:20<03:17,  5.65s/it]

...

Epoch  40/50 | Loss: 39596.3008 | Attr Loss: 8579.1436 | Struct Loss: 163664.9219 | ROC AUC: 0.8946


 90%|█████████ | 45/50 [03:55<00:27,  5.48s/it]

Epoch  45/50 | Loss: 39581.6953 | Attr Loss: 8567.2461 | Struct Loss: 163639.4844 | ROC AUC: 0.8946


100%|██████████| 50/50 [04:21<00:00,  5.23s/it]

Epoch  50/50 | Loss: 39537.8594 | Attr Loss: 8568.3301 | Struct Loss: 163415.9688 | ROC AUC: 0.8947


In [12]:
print("Results:\n")
print(f"Total Loss: {loss_history[-1]:.4f}")
print(f"Attribute Loss: {attr_loss_history[-1]:.4f}")
print(f"Structure Loss: {struct_loss_history[-1]:.4f}")
print(f"ROC AUC: {roc_auc_history[-1]:.4f}")

Results:

Total Loss: 39537.8594
Attribute Loss: 8568.3301
Structure Loss: 163415.9688
ROC AUC: 0.8947
